# EvolveAI Brain v3.2 — Evidence-backed extraction

This notebook reads the existing Brain v3.1 semantic document store. It does not retrain the SNN, overwrite v3.1 files, or treat extracted candidates as verified facts. Every candidate retains its source document, source IDs, domain, and exact evidence sentence.

In [ ]:
from pathlib import Path
import subprocess, sys
REPO = Path('/content/EvolveAI')
BRANCH = 'evolveai-maxgpu-one-shot'
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'https://github.com/swasthik-s/EvolveAI.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH],check=True)
SNN = REPO/'snn'
sys.path.insert(0,str(SNN))
print('Repository:',REPO)
print('Extractor:',SNN/'brain'/'evidence_extractor.py')


In [ ]:
import subprocess
result = subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_evidence_extractor.py','-v'],cwd=SNN,capture_output=True,text=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError('Brain v3.2 extractor tests failed. Stop and inspect the traceback.')
print('Brain v3.2 unit tests: PASS')


## Run extraction on the existing v3.1 index

Mount Google Drive if it is not already mounted. Output is written to a separate `v3_2_extraction` folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
MEMORY_DIR = Path('/content/drive/MyDrive/EvolveAI/brain_v3_memory')
DOCS = MEMORY_DIR/'semantic_documents.jsonl'
if not DOCS.is_file():
    raise FileNotFoundError(f'Brain v3.1 document store not found: {DOCS}')
from brain.evidence_extractor import run_extraction
report = run_extraction(MEMORY_DIR)
import json
print(json.dumps(report,indent=2))


In [ ]:
OUT = MEMORY_DIR/'v3_2_extraction'
print('Candidate relation examples:')
relation_file = OUT/'candidate_relations.jsonl'
shown = 0
if relation_file.exists():
    with relation_file.open(encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            print(json.dumps(item,ensure_ascii=False))
            shown += 1
            if shown == 10: break
print('Candidate entity examples:')
entity_file = OUT/'candidate_entities.jsonl'
shown = 0
if entity_file.exists():
    with entity_file.open(encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            print(json.dumps(item,ensure_ascii=False))
            shown += 1
            if shown == 10: break


## Interpretation and limitations

- These are heuristic candidates, not verified facts.
- Relation candidates are emitted only when an explicit text pattern matches.
- Every output record retains the exact evidence sentence, source document ID, source IDs, and domain.
- The extractor does not infer causal or semantic links from co-occurrence counts.
- Review precision and false positives before integrating candidates into the long-term knowledge graph.